# 🚀 Multimodal RAG — Open WebUI + Ollama + ChromaDB

**Full pipeline: PDF → Text + Images + Tables → CLIP/mxbai Embeddings → ChromaDB → llama3.2 answers**

```
📁 Your Documents
       │
       ├── 📝 Text    → chunk (300w/50 overlap) → mxbai-embed-large  ─┐
       ├── 📊 Tables  → markdown text           → mxbai-embed-large  ─┼─→ ChromaDB → Rerank → llama3.2
       └── 🖼️ Images  → CLIP ViT-B/32 visual   → 512-dim embedding  ─┘
```

### What you get
- Documents-only answers (no hallucination from model knowledge)
- Image semantic search via CLIP (cross-modal: text query finds matching images)
- Table-aware retrieval (tables converted to structured text, embedded + retrieved)
- FastAPI server that Open WebUI connects to as an OpenAI-compatible endpoint
- Page-level citations in every answer

### Models used
| Role | Model | Why |
|------|-------|-----|
| Text/table embedding | `mxbai-embed-large` | Best open embedding, 1024-dim |
| Image embedding | `CLIP ViT-B/32` | Fast local visual embeddings |
| Reranker | `cross-encoder/ms-marco-MiniLM-L-6-v2` | Fast + accurate reranking |
| Answer LLM | `llama3.2` | Already installed, great at RAG |
| Vector DB | `ChromaDB` | Open-source, used by Open WebUI |


## 📦 CELL 1 — Install All Dependencies

Run this once. Takes 3-5 minutes depending on your internet speed.

In [1]:
# ─── Install all required libraries ───────────────────────────────────────────
# chromadb          → vector database
# pymupdf (fitz)    → PDF text + image extraction
# pdfplumber        → table extraction (text-based PDFs)
# camelot-py[cv]    → lattice/bordered table extraction
# open-clip-torch   → CLIP image embeddings (local, no API)
# sentence-transformers → CrossEncoder reranker
# fastapi + uvicorn → serve OpenAI-compatible API for Open WebUI
# pillow            → image processing
# ollama            → talk to local Ollama models
# tqdm              → progress bars

!pip install -q \
    chromadb \
    pymupdf \
    pdfplumber \
    "camelot-py[cv]" \
    open-clip-torch \
    torch torchvision \
    sentence-transformers \
    fastapi \
    "uvicorn[standard]" \
    pillow \
    ollama \
    tqdm \
    opencv-python-headless

print("✅ All dependencies installed!")

✅ All dependencies installed!



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## ⚙️ CELL 2 — Configuration

**Change `DOCS_PATH` to your document folder.** Everything else can stay as default.

In [2]:
import os

# ══════════════════════════════════════════════════════════════════════════════
#  ⚙️  CONFIGURATION — Edit these values
# ══════════════════════════════════════════════════════════════════════════════

# 📁 Path to your folder containing PDFs
# Example: DOCS_PATH = r"C:\Users\LENOVO\Documents\MyDocs"
DOCS_PATH = r"C:\Users\LENOVO\Desktop\Multimodal AI Store\Data\Documets"  # ← CHANGE THIS to your documents folder path

# 🗄️ Where to store ChromaDB on disk
CHROMA_PATH = r"C:\Users\LENOVO\Desktop\Multimodal AI Store\Data\chroma_db"

# 🖼️ Where to save extracted images
IMAGES_PATH = r"C:\Users\LENOVO\Desktop\Multimodal AI Store\Data\images\extracted_images"

# 🤖 Ollama models
EMBED_MODEL   = "mxbai-embed-large"   # text + table embeddings
ANSWER_MODEL  = "llama3.2"            # final answer generation

# ✂️ Text chunking parameters
CHUNK_SIZE    = 300   # words per chunk
CHUNK_OVERLAP = 50    # overlapping words between chunks

# 🔍 Retrieval parameters
TOP_K_RETRIEVE = 8    # how many chunks to retrieve before reranking
TOP_K_RERANK   = 5    # how many chunks to keep after reranking

# 🌐 FastAPI server settings (for Open WebUI)
API_HOST = "0.0.0.0"
API_PORT = 8100

# ══════════════════════════════════════════════════════════════════════════════

# Create required directories
os.makedirs(DOCS_PATH, exist_ok=True)
os.makedirs(CHROMA_PATH, exist_ok=True)
os.makedirs(IMAGES_PATH, exist_ok=True)

print("✅ Configuration set!")
print(f"   📁 Documents : {os.path.abspath(DOCS_PATH)}")
print(f"   🗄️  ChromaDB  : {os.path.abspath(CHROMA_PATH)}")
print(f"   🖼️  Images    : {os.path.abspath(IMAGES_PATH)}")

✅ Configuration set!
   📁 Documents : C:\Users\LENOVO\Desktop\Multimodal AI Store\Data\Documets
   🗄️  ChromaDB  : C:\Users\LENOVO\Desktop\Multimodal AI Store\Data\chroma_db
   🖼️  Images    : C:\Users\LENOVO\Desktop\Multimodal AI Store\Data\images\extracted_images


## 🔧 CELL 3 — Load All Models

Loads CLIP and the reranker into memory once. These stay loaded for the entire session.

In [3]:
import torch
import open_clip
from sentence_transformers import CrossEncoder
import warnings
warnings.filterwarnings("ignore")

# ── CLIP model for IMAGE embeddings ───────────────────────────────────────────
# ViT-B/32 is the fastest CLIP variant — good balance of speed vs quality
# It produces 512-dimensional embeddings
# KEY FEATURE: CLIP is cross-modal — a text query like "bar chart showing revenue"
# can find matching images even without any text description of those images!

print("Loading CLIP ViT-B/32...")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai"  # uses OpenAI's pretrained weights (downloaded automatically)
)
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")
clip_model.eval()

# Use GPU if available, else CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = clip_model.to(DEVICE)
print(f"✅ CLIP loaded on {DEVICE}")

# ── CrossEncoder Reranker ─────────────────────────────────────────────────────
# After retrieving top-K chunks from ChromaDB (fast vector search),
# the reranker does a deep semantic comparison of query vs each chunk.
# This improves precision significantly — especially for long documents.

print("Loading CrossEncoder reranker...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("✅ Reranker loaded")

print(f"\n🎉 All models ready! Device: {DEVICE}")

Loading CLIP ViT-B/32...
✅ CLIP loaded on cpu
Loading CrossEncoder reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Reranker loaded

🎉 All models ready! Device: cpu


## 📝 CELL 4 — Text Extraction + Chunking

Reads all PDFs in your folder. Splits each page into overlapping 300-word chunks so context isn't lost at boundaries.

In [4]:
import fitz  # PyMuPDF
from pathlib import Path
from tqdm import tqdm

def extract_text_from_pdf(pdf_path):
    """
    Extract clean text from a PDF, page by page.
    Returns list of {text, page_num, source_file} dicts.
    """
    doc = fitz.open(str(pdf_path))
    pages = []
    for i, page in enumerate(doc):
        # "text" mode gives clean plain text
        # "blocks" mode gives positional info — we use plain here
        text = page.get_text("text").strip()
        if len(text) > 50:  # skip near-empty pages
            pages.append({
                "text": text,
                "page_num": i + 1,
                "source": pdf_path.name
            })
    doc.close()
    return pages


def chunk_page(page_data, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """
    Split a page's text into overlapping word-level chunks.

    Why overlap? Without overlap, a sentence split across chunk boundaries
    would be meaningless in both chunks. With 50-word overlap, the context
    from the previous chunk carries forward.

    Example:
      chunk_size=300, overlap=50 → step=250 words per chunk
      Page with 600 words → chunks at word 0, 250, 500
    """
    words = page_data["text"].split()
    if not words:
        return []

    step = chunk_size - overlap
    chunks = []
    for i in range(0, len(words), step):
        chunk_words = words[i : i + chunk_size]
        if len(chunk_words) < 15:  # skip tiny trailing fragments
            continue
        chunks.append({
            "content": " ".join(chunk_words),
            "page": page_data["page_num"],
            "source": page_data["source"],
            "type": "text",
            "chunk_index": i // step
        })
    return chunks


def load_all_text_chunks(docs_path):
    """Process all PDFs in the folder and return all text chunks."""
    docs_folder = Path(docs_path)
    pdf_files = list(docs_folder.glob("*.pdf"))

    if not pdf_files:
        print(f"⚠️  No PDF files found in: {docs_folder.resolve()}")
        print("    Please add PDF files to that folder and re-run this cell.")
        return []

    print(f"Found {len(pdf_files)} PDF(s):")
    for f in pdf_files:
        print(f"   • {f.name}")

    all_chunks = []
    for pdf_path in tqdm(pdf_files, desc="Extracting text"):
        pages = extract_text_from_pdf(pdf_path)
        for page in pages:
            all_chunks.extend(chunk_page(page))

    print(f"\n✅ Text extraction complete!")
    print(f"   Total chunks: {len(all_chunks)}")
    return all_chunks


# Run it
text_chunks = load_all_text_chunks(DOCS_PATH)

# Preview first chunk
if text_chunks:
    print(f"\n📋 Sample chunk from '{text_chunks[0]['source']}' page {text_chunks[0]['page']}:")
    print("-" * 60)
    print(text_chunks[0]['content'][:300] + "...")

Found 2 PDF(s):
   • Attention all you need.pdf
   • Sample.pdf


Extracting text: 100%|████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  4.12it/s]


✅ Text extraction complete!
   Total chunks: 38

📋 Sample chunk from 'Attention all you need.pdf' page 1:
------------------------------------------------------------
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Par...


## 📊 CELL 5 — Table Extraction

Tables are converted to a structured text format (`col1 | col2 | col3`) so they can be embedded and retrieved like any other text chunk. When your question matches a table, you get the full table data in the answer.

In [5]:
import pdfplumber
from pathlib import Path

def table_to_text(table, header=True):
    """
    Convert a 2D table list into pipe-separated markdown text.

    Input:  [["Name", "Age"], ["Alice", "30"], ["Bob", "25"]]
    Output: "Name | Age\nAlice | 30\nBob | 25"

    This format is:
    - Embeddable as text
    - Readable by the LLM
    - Preserves column relationships
    """
    rows = []
    for i, row in enumerate(table):
        # Clean None values and strip whitespace
        cleaned = [str(cell).strip() if cell is not None else "" for cell in row]
        rows.append(" | ".join(cleaned))
        # Add markdown separator after header row
        if header and i == 0:
            rows.append(" | ".join(["---"] * len(row)))
    return "\n".join(rows)


def extract_tables_from_pdf(pdf_path):
    """
    Extract all tables from a PDF using pdfplumber.
    pdfplumber uses line detection + bounding boxes to find tables.
    Works best for PDFs with visible table borders or well-structured layouts.
    """
    table_chunks = []
    try:
        with pdfplumber.open(str(pdf_path)) as pdf:
            for i, page in enumerate(pdf.pages):
                tables = page.extract_tables()
                for t_idx, table in enumerate(tables):
                    if not table or len(table) < 2:
                        continue  # skip single-row non-tables
                    table_text = table_to_text(table)
                    if len(table_text.strip()) < 20:
                        continue
                    # Prefix with [TABLE] so the LLM understands the context
                    content = f"[TABLE from page {i+1}]\n{table_text}"
                    table_chunks.append({
                        "content": content,
                        "page": i + 1,
                        "source": Path(pdf_path).name,
                        "type": "table",
                        "table_index": t_idx
                    })
    except Exception as e:
        print(f"⚠️  Table extraction failed for {pdf_path}: {e}")
    return table_chunks


def load_all_table_chunks(docs_path):
    """Extract tables from all PDFs in the folder."""
    docs_folder = Path(docs_path)
    pdf_files = list(docs_folder.glob("*.pdf"))
    all_table_chunks = []
    for pdf_path in tqdm(pdf_files, desc="Extracting tables"):
        chunks = extract_tables_from_pdf(pdf_path)
        all_table_chunks.extend(chunks)
    print(f"✅ Tables extracted: {len(all_table_chunks)} table chunks")
    return all_table_chunks


# Run it
table_chunks = load_all_table_chunks(DOCS_PATH)

# Preview a table if found
if table_chunks:
    print(f"\n📊 Sample table from '{table_chunks[0]['source']}' page {table_chunks[0]['page']}:")
    print("-" * 60)
    print(table_chunks[0]['content'][:500])

Extracting tables: 100%|██████████████████████████████████████████████████████████████████████████████| 2/2 [00:13<00:00,  6.72s/it]

✅ Tables extracted: 9 table chunks

📊 Sample table from 'Attention all you need.pdf' page 9:
------------------------------------------------------------
[TABLE from page 9]
train
N d d h d d P ϵ
model ff k v drop ls steps
---
6 512 2048 8 64 64 0.1 0.1 100K
1 512 512
4 128 128
16 32 32
32 16 16
16
32
2
4
8
256 32 32
1024 128 128
1024
4096
0.0
0.2
0.0
0.2
positionalembeddinginsteadofsinusoids
6 1024 4096 16 0.3 300K


## 🖼️ CELL 6 — Image Extraction + CLIP Embedding

**This is the key innovation.** Instead of using LLaVA to describe each image (slow!), we:
1. Extract each image from the PDF and save it as a PNG
2. Run CLIP on it → get a 512-dim visual embedding
3. Store the embedding in ChromaDB

When you ask a question, we also CLIP-encode your text query → find visually similar images.
CLIP understands concepts like "bar chart", "network diagram", "person", "logo" without any image captions!

In [6]:
from PIL import Image
import fitz
import torch

def get_clip_image_embedding(image_path):
    """
    Generate a 512-dim CLIP embedding from an image file.

    CLIP (Contrastive Language-Image Pretraining) maps both images
    and text into the SAME embedding space. This means:
      embed("a pie chart") ≈ embed(image_of_pie_chart)

    That's what enables text-to-image search with no captions!
    """
    image = Image.open(image_path).convert("RGB")
    tensor = clip_preprocess(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        emb = clip_model.encode_image(tensor)
        emb = emb / emb.norm(dim=-1, keepdim=True)  # L2 normalize for cosine similarity
    return emb[0].cpu().numpy().tolist()


def get_clip_text_embedding(text):
    """
    Generate a 512-dim CLIP embedding from text.
    Used at query time to search the image collection.
    """
    tokens = clip_tokenizer([text[:77]]).to(DEVICE)  # CLIP has 77-token limit
    with torch.no_grad():
        emb = clip_model.encode_text(tokens)
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb[0].cpu().numpy().tolist()


def extract_images_from_pdf(pdf_path, output_dir, min_size_kb=5):
    """
    Extract all embedded images from a PDF.

    min_size_kb=5: Skip tiny images (bullets, icons, page decorations).
    Images smaller than 5KB are almost never meaningful content.

    For each image:
    1. Save to disk as PNG
    2. Generate CLIP embedding
    3. Return as a chunk with metadata
    """
    doc = fitz.open(str(pdf_path))
    image_chunks = []
    pdf_name = Path(pdf_path).stem

    for page_num, page in enumerate(doc):
        images = page.get_images(full=True)
        for img_idx, img_info in enumerate(images):
            xref = img_info[0]
            try:
                base_image = doc.extract_image(xref)
                image_bytes = base_image["image"]

                # Skip tiny images
                if len(image_bytes) < min_size_kb * 1024:
                    continue

                # Save to disk
                img_filename = f"{pdf_name}_page{page_num+1}_img{img_idx}.png"
                img_path = os.path.join(output_dir, img_filename)
                with open(img_path, "wb") as f:
                    f.write(image_bytes)

                # Generate CLIP embedding
                clip_emb = get_clip_image_embedding(img_path)

                # The "content" is a text description for display purposes
                # The actual retrieval uses the CLIP embedding, not this text
                content = f"[IMAGE from {Path(pdf_path).name}, page {page_num+1}, image {img_idx+1}]"

                image_chunks.append({
                    "content": content,
                    "page": page_num + 1,
                    "source": Path(pdf_path).name,
                    "type": "image",
                    "image_path": img_path,
                    "clip_embedding": clip_emb
                })

            except Exception as e:
                pass  # skip broken/unsupported image formats

    doc.close()
    return image_chunks


def load_all_image_chunks(docs_path, output_dir):
    """Extract and embed all images from all PDFs."""
    docs_folder = Path(docs_path)
    pdf_files = list(docs_folder.glob("*.pdf"))
    all_image_chunks = []
    for pdf_path in tqdm(pdf_files, desc="Extracting images"):
        chunks = extract_images_from_pdf(pdf_path, output_dir)
        all_image_chunks.extend(chunks)
    print(f"✅ Images processed: {len(all_image_chunks)} image chunks with CLIP embeddings")
    return all_image_chunks


# Run it
image_chunks = load_all_image_chunks(DOCS_PATH, IMAGES_PATH)

Extracting images: 100%|██████████████████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.31s/it]

✅ Images processed: 4 image chunks with CLIP embeddings


## 🧠 CELL 7 — Text + Table Embeddings via mxbai-embed-large

In [7]:
import ollama

def get_mxbai_embedding(text):
    """
    Generate a 1024-dim text embedding using mxbai-embed-large via Ollama.

    mxbai-embed-large is one of the best open-source embedding models.
    It's specifically trained for retrieval tasks (not just similarity).

    We cap text at 4000 chars to stay within the model's context window.
    For typical 300-word chunks, this is never an issue.
    """
    text = str(text)[:1500]
    try:
        response = ollama.embed(
            model="mxbai-embed-large",
            input=text
        )
        return response["embeddings"][0]
    except Exception as e:
        print(f"⚠️  Embedding error: {e}")
        return None


# Sanity check — verify Ollama is running and mxbai is available
print("Testing mxbai-embed-large...")
test_emb = get_mxbai_embedding("This is a test sentence for embedding.")
if test_emb:
    print(f"✅ mxbai-embed-large working!")
    print(f"   Embedding dimensions: {len(test_emb)}")
else:
    print("❌ Embedding failed. Make sure Ollama is running:")
    print("   Open a terminal and run: ollama serve")
    print("   Also check: ollama list  (should show mxbai-embed-large)")

Testing mxbai-embed-large...
✅ mxbai-embed-large working!
   Embedding dimensions: 1024


## 🗄️ CELL 8 — Store Everything in ChromaDB

We use **two separate ChromaDB collections** because text (1024-dim) and images (512-dim) have different embedding dimensions and cannot be mixed in one collection.

In [8]:
import chromadb
from tqdm import tqdm

# ── Initialize ChromaDB (persistent on disk) ──────────────────────────────────
# PersistentClient saves to disk so you don't need to re-embed every run.
# Just run this notebook once for ingestion, then use the query cells.

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# Collection 1: Text + Tables (mxbai-embed-large → 1024 dims)
# cosine distance is best for embedding similarity tasks
text_collection = chroma_client.get_or_create_collection(
    name="text_and_tables",
    metadata={"hnsw:space": "cosine"}  # cosine similarity for semantic search
)

# Collection 2: Images (CLIP ViT-B/32 → 512 dims)
image_collection = chroma_client.get_or_create_collection(
    name="images",
    metadata={"hnsw:space": "cosine"}
)

print(f"📊 Current ChromaDB state:")
print(f"   text_and_tables collection: {text_collection.count()} chunks")
print(f"   images collection: {image_collection.count()} chunks")

# ── Store Text Chunks ─────────────────────────────────────────────────────────
all_text_table_chunks = text_chunks + table_chunks
print(f"\nEmbedding {len(all_text_table_chunks)} text/table chunks...")
print("(This may take a few minutes for large documents)")

BATCH_SIZE = 20  # embed in batches to avoid timeouts
stored_count = 0

for i in tqdm(range(0, len(all_text_table_chunks), BATCH_SIZE), desc="Storing text/tables"):
    batch = all_text_table_chunks[i : i + BATCH_SIZE]
    ids, embeddings, documents, metadatas = [], [], [], []

    for j, chunk in enumerate(batch):
        emb = get_mxbai_embedding(chunk["content"])
        if emb is None:
            continue
        chunk_id = f"{chunk['type']}_{chunk['source']}_{chunk['page']}_{i+j}"
        ids.append(chunk_id)
        embeddings.append(emb)
        documents.append(chunk["content"])
        metadatas.append({
            "page": chunk["page"],
            "type": chunk["type"],
            "source": chunk["source"]
        })

    if ids:  # only add if we have valid embeddings
        text_collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=documents,
            metadatas=metadatas
        )
        stored_count += len(ids)

print(f"✅ Text/tables stored: {stored_count} chunks")

# ── Store Image Chunks ────────────────────────────────────────────────────────
print(f"\nStoring {len(image_chunks)} image embeddings...")
img_stored = 0

for i, chunk in enumerate(tqdm(image_chunks, desc="Storing images")):
    img_id = f"img_{chunk['source']}_{chunk['page']}_{i}"
    try:
        image_collection.add(
            ids=[img_id],
            embeddings=[chunk["clip_embedding"]],
            documents=[chunk["content"]],
            metadatas=[{
                "page": chunk["page"],
                "type": "image",
                "source": chunk["source"],
                "image_path": chunk["image_path"]
            }]
        )
        img_stored += 1
    except Exception as e:
        pass

print(f"✅ Images stored: {img_stored} chunks")

print(f"""
🎉 Ingestion complete!
   text_and_tables : {text_collection.count()} chunks
   images          : {image_collection.count()} chunks
   Total           : {text_collection.count() + image_collection.count()} chunks
""")

📊 Current ChromaDB state:
   text_and_tables collection: 44 chunks
   images collection: 4 chunks

Embedding 47 text/table chunks...
(This may take a few minutes for large documents)


Storing text/tables: 100%|███████████████████████████████████████████████████████████████████████████| 3/3 [06:09<00:00, 123.22s/it]


✅ Text/tables stored: 47 chunks

Storing 4 image embeddings...


Storing images: 100%|█████████████████████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  2.88it/s]


✅ Images stored: 4 chunks

🎉 Ingestion complete!
   text_and_tables : 47 chunks
   images          : 4 chunks
   Total           : 51 chunks



## 🔍 CELL 9 — Retrieval + Reranking Functions

**Two-stage retrieval:**
1. **Stage 1 (Fast):** ChromaDB HNSW vector search → top 8 candidates
2. **Stage 2 (Precise):** CrossEncoder reranker → top 5 final chunks

This is the same pattern used by production RAG systems.

In [9]:
def retrieve_text_chunks(query, k=TOP_K_RETRIEVE):
    """
    Stage 1: Fast approximate nearest-neighbor search for text/tables.
    ChromaDB uses HNSW index → O(log n) search, very fast.
    Returns top-k most similar chunks by cosine distance.
    """
    query_emb = get_mxbai_embedding(query)
    if not query_emb:
        return []

    if text_collection.count() == 0:
        return []

    results = text_collection.query(
        query_embeddings=[query_emb],
        n_results=min(k, text_collection.count())
    )

    chunks = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        chunks.append({
            "content": doc,
            "page": meta["page"],
            "type": meta["type"],
            "source": meta["source"],
            "distance": dist  # lower = more similar
        })
    return chunks


def retrieve_image_chunks(query, k=2):
    """
    Cross-modal image search: encode query text with CLIP,
    search image collection.

    Example: query="revenue bar chart" finds bar chart images
    even though we never wrote a text description of those images!
    """
    if image_collection.count() == 0:
        return []

    clip_emb = get_clip_text_embedding(query)
    results = image_collection.query(
        query_embeddings=[clip_emb],
        n_results=min(k, image_collection.count())
    )

    chunks = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        # Only include images with high visual similarity (dist < 0.5)
        if dist < 0.5:
            chunks.append({
                "content": doc,
                "page": meta["page"],
                "type": "image",
                "source": meta.get("source", ""),
                "image_path": meta.get("image_path", ""),
                "distance": dist
            })
    return chunks


def rerank_chunks(query, text_chunks, top_k=TOP_K_RERANK):
    """
    Stage 2: CrossEncoder reranking.

    The CrossEncoder reads both the query AND each chunk together
    (not independently like bi-encoders) — this gives much better
    relevance scores but is slower. We only run it on top-k candidates.

    Image chunks are not reranked (they already have CLIP similarity scores)
    and are always passed through.
    """
    if not text_chunks:
        return []

    pairs = [[query, chunk["content"]] for chunk in text_chunks]
    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(text_chunks, scores),
        key=lambda x: x[1],
        reverse=True  # higher score = more relevant
    )
    return [chunk for chunk, score in ranked[:top_k]]


def full_retrieve(query):
    """
    Complete retrieval pipeline:
    1. Get candidate text/table chunks from ChromaDB
    2. Rerank with CrossEncoder
    3. Get matching images via CLIP
    4. Return combined results
    """
    # Text + table retrieval
    candidates = retrieve_text_chunks(query, k=TOP_K_RETRIEVE)
    top_text   = rerank_chunks(query, candidates, top_k=TOP_K_RERANK)

    # Image retrieval (CLIP cross-modal)
    top_images = retrieve_image_chunks(query, k=2)

    return top_text, top_images


print("✅ Retrieval functions ready!")

# Quick test
if text_collection.count() > 0:
    test_query = "What is this document about?"
    t, i = full_retrieve(test_query)
    print(f"\n🧪 Test retrieval for: '{test_query}'")
    print(f"   Found {len(t)} text/table chunks, {len(i)} image chunks")
    if t:
        print(f"   Top result from page {t[0]['page']} ({t[0]['type']}):")
        print(f"   {t[0]['content'][:200]}...")

✅ Retrieval functions ready!

🧪 Test retrieval for: 'What is this document about?'
   Found 5 text/table chunks, 0 image chunks
   Top result from page 14 (table):
   [TABLE from page 14]
 |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  | 
--- ...


## 🤖 CELL 10 — Answer Generation with llama3.2

The LLM is instructed to answer **only from the provided context** — no hallucination from its training data.

In [10]:
def build_prompt(query, text_chunks, image_chunks):
    """
    Build a RAG prompt that:
    1. Gives the LLM context from retrieved chunks
    2. Tells it to answer ONLY from that context
    3. Requires page number citations
    4. Mentions image references if found
    """
    # Build context string from top text/table chunks
    context_parts = []
    for chunk in text_chunks:
        label = f"[{chunk['type'].upper()} | Source: {chunk['source']} | Page {chunk['page']}]"
        context_parts.append(f"{label}\n{chunk['content']}")

    context = "\n\n" + ("\n\n---\n\n".join(context_parts))[:4000]  # cap total context

    # Add image references to context
    image_ref_text = ""
    if image_chunks:
        img_refs = []
        for img in image_chunks:
            img_refs.append(f"  • {img['source']}, Page {img['page']}: {img['image_path']}")
        image_ref_text = f"""

RELEVANT IMAGES FOUND:
{chr(10).join(img_refs)}
(These images from the document may be relevant to the question)"""

    prompt = f"""You are a precise document assistant. Answer ONLY based on the context provided below.
Do NOT use any outside knowledge. If the answer is not in the context, say exactly:
"I could not find this information in the provided documents."

Always cite the source and page number for each piece of information, like: (Source: filename.pdf, Page X)

=== DOCUMENT CONTEXT ==={context}{image_ref_text}
=== END CONTEXT ===

Question: {query}

Answer (cite pages):"""

    return prompt


def ask(question, stream=True):
    """
    Full RAG pipeline:
    query → retrieve → rerank → prompt → llama3.2 → answer

    stream=True prints the answer token by token as it's generated.
    stream=False returns the full answer string.
    """
    print(f"\n{'═'*70}")
    print(f"❓ Question: {question}")
    print(f"{'═'*70}")

    # Step 1: Retrieve
    print("🔍 Retrieving relevant chunks...")
    top_text, top_images = full_retrieve(question)
    print(f"   ✓ {len(top_text)} text/table chunks, {len(top_images)} images")

    if not top_text and not top_images:
        return "❌ No relevant content found in the documents."

    # Show what was retrieved (for transparency)
    for chunk in top_text:
        print(f"   → [{chunk['type']}] {chunk['source']}, Page {chunk['page']}")
    for img in top_images:
        print(f"   → [IMAGE] {img['source']}, Page {img['page']}")

    # Step 2: Build prompt
    prompt = build_prompt(question, top_text, top_images)

    # Step 3: Generate answer
    print(f"\n💬 Answer:")
    print("-" * 70)

    if stream:
        full_answer = ""
        for chunk in ollama.chat(
            model=ANSWER_MODEL,
            messages=[{"role": "user", "content": prompt}],
            stream=True
        ):
            token = chunk["message"]["content"]
            print(token, end="", flush=True)
            full_answer += token
        print()  # newline after streaming
        return full_answer
    else:
        response = ollama.chat(
            model=ANSWER_MODEL,
            messages=[{"role": "user", "content": prompt}]
        )
        answer = response["message"]["content"]
        print(answer)
        return answer


print("✅ Answer generation ready!")

✅ Answer generation ready!


## 🧪 CELL 11 — Test Your RAG System

Ask any question about your documents. Change the queries below to match your document content.

In [11]:
# ── Test queries — change these to match your documents ───────────────────────

# Question 1: General topic
ask("What is the main topic of these documents?")


══════════════════════════════════════════════════════════════════════
❓ Question: What is the main topic of these documents?
══════════════════════════════════════════════════════════════════════
🔍 Retrieving relevant chunks...
   ✓ 5 text/table chunks, 0 images
   → [text] Attention all you need.pdf, Page 12
   → [table] Attention all you need.pdf, Page 15
   → [table] Attention all you need.pdf, Page 14
   → [table] Attention all you need.pdf, Page 13
   → [table] Attention all you need.pdf, Page 13

💬 Answer:
----------------------------------------------------------------------
I could not find this information in the provided documents.


'I could not find this information in the provided documents.'

In [12]:
# Question 2: Table-specific (will find table chunks)
ask("What data is shown in the tables?")


══════════════════════════════════════════════════════════════════════
❓ Question: What data is shown in the tables?
══════════════════════════════════════════════════════════════════════
🔍 Retrieving relevant chunks...
   ✓ 5 text/table chunks, 0 images
   → [text] Attention all you need.pdf, Page 9
   → [table] Attention all you need.pdf, Page 15
   → [table] Attention all you need.pdf, Page 14
   → [table] Attention all you need.pdf, Page 13
   → [table] Attention all you need.pdf, Page 13

💬 Answer:
----------------------------------------------------------------------
The tables appear to contain various symbols and words that are not clearly related to the provided document context. However, I found a section on page 14 that mentions "ehT", "waL", etc., which seem to be some kind of encoding or representation.

Additionally, on page 15, there is a list of words in reverse order, starting with ">SOE<" and ending with "<dap<".


'The tables appear to contain various symbols and words that are not clearly related to the provided document context. However, I found a section on page 14 that mentions "ehT", "waL", etc., which seem to be some kind of encoding or representation.\n\nAdditionally, on page 15, there is a list of words in reverse order, starting with ">SOE<" and ending with "<dap<".'

In [13]:
# Question 3: Image-specific (CLIP will find matching images)
ask("Are there any charts, diagrams or figures in the document?")


══════════════════════════════════════════════════════════════════════
❓ Question: Are there any charts, diagrams or figures in the document?
══════════════════════════════════════════════════════════════════════
🔍 Retrieving relevant chunks...
   ✓ 5 text/table chunks, 0 images
   → [table] Attention all you need.pdf, Page 15
   → [table] Attention all you need.pdf, Page 14
   → [table] Attention all you need.pdf, Page 13
   → [table] Attention all you need.pdf, Page 13
   → [table] Attention all you need.pdf, Page 15

💬 Answer:
----------------------------------------------------------------------


KeyboardInterrupt: 

In [14]:
print("Text count:", text_collection.count())
print("Image count:", image_collection.count())

Text count: 47
Image count: 4


## 🌐 CELL 12 — FastAPI Server (Open WebUI Integration)

This creates an **OpenAI-compatible API** that Open WebUI connects to.

Open WebUI thinks it's talking to OpenAI — but it's actually talking to your RAG system!

**After running this cell:**
1. Open WebUI → Settings → Connections
2. Add new OpenAI connection:
   - URL: `http://127.0.0.1:8100/v1`
   - API Key: `rag-key` (or any string)
3. Select model `rag-llama3.2`
4. Chat! Every message searches your documents first.

In [15]:
# Save the FastAPI server as a separate Python file
# We run it in a background thread so the notebook stays interactive

server_code = '''
import sys, os
sys.path.insert(0, os.getcwd())

from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from typing import List, Optional
import json, time, uuid
import ollama
import chromadb
import torch
import open_clip
from sentence_transformers import CrossEncoder
import warnings
warnings.filterwarnings("ignore")

# ── Config (must match your notebook config) ──────────────────────────────────
CHROMA_PATH   = r"C:\Users\LENOVO\Desktop\Multimodal AI Store\Data\chroma_db"
EMBED_MODEL   = "mxbai-embed-large"
ANSWER_MODEL  = "llama3.2"
TOP_K_RETRIEVE = 8
TOP_K_RERANK   = 5

# ── Load models ───────────────────────────────────────────────────────────────
print("Loading CLIP...")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")
clip_model.eval()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = clip_model.to(DEVICE)

print("Loading reranker...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Connecting to ChromaDB...")
chroma_client   = chromadb.PersistentClient(path=CHROMA_PATH)
text_collection = chroma_client.get_collection("text_and_tables")
image_collection = chroma_client.get_collection("images")
print(f"✅ Connected — {text_collection.count()} text chunks, {image_collection.count()} image chunks")

# ── Embedding functions ───────────────────────────────────────────────────────
def get_mxbai_embedding(text):
    r = ollama.embed(model=EMBED_MODEL, input=str(text)[:4000])
    return r["embeddings"][0]

def get_clip_text_embedding(text):
    tokens = clip_tokenizer([text[:77]]).to(DEVICE)
    with torch.no_grad():
        emb = clip_model.encode_text(tokens)
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb[0].cpu().numpy().tolist()

# ── Retrieval ─────────────────────────────────────────────────────────────────
def retrieve_and_answer(question):
    # Text retrieval
    q_emb = get_mxbai_embedding(question)
    t_res = text_collection.query(query_embeddings=[q_emb], n_results=min(TOP_K_RETRIEVE, text_collection.count()))
    text_chunks = [
        {"content": d, "page": m["page"], "type": m["type"], "source": m["source"]}
        for d, m in zip(t_res["documents"][0], t_res["metadatas"][0])
    ]

    # Rerank
    if text_chunks:
        pairs  = [[question, c["content"]] for c in text_chunks]
        scores = reranker.predict(pairs)
        text_chunks = [c for c, _ in sorted(zip(text_chunks, scores), key=lambda x: x[1], reverse=True)[:TOP_K_RERANK]]

    # Image retrieval
    image_chunks = []
    if image_collection.count() > 0:
        ci_emb = get_clip_text_embedding(question)
        i_res  = image_collection.query(query_embeddings=[ci_emb], n_results=min(2, image_collection.count()))
        image_chunks = [
            {"content": d, "page": m["page"], "source": m.get("source",""), "image_path": m.get("image_path","")}
            for d, m, dist in zip(i_res["documents"][0], i_res["metadatas"][0], i_res["distances"][0])
            if dist < 0.5
        ]

    # Build prompt
    ctx_parts = [
        f"[{c[\"type\"].upper()} | {c[\"source\"]} | Page {c[\"page\"]}]\\n{c[\"content\"]}"
        for c in text_chunks
    ]
    context = "\\n\\n---\\n\\n".join(ctx_parts)[:4000]

    img_note = ""
    if image_chunks:
        img_note = "\\n\\nRELEVANT IMAGES: " + ", ".join([f"{i[\"source\"]} p.{i[\"page\"]}" for i in image_chunks])

    prompt = f"""Answer ONLY from the context below. Cite page numbers. If not found, say so.

CONTEXT:
{context}{img_note}

Question: {question}
Answer:"""

    return prompt

# ── FastAPI app ───────────────────────────────────────────────────────────────
app = FastAPI(title="Multimodal RAG API")

class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: str
    messages: List[Message]
    stream: Optional[bool] = False
    temperature: Optional[float] = 0.1

@app.get("/v1/models")
def list_models():
    return {
        "object": "list",
        "data": [{"id": "rag-llama3.2", "object": "model", "owned_by": "local-rag"}]
    }

@app.post("/v1/chat/completions")
async def chat(req: ChatRequest):
    # Extract the latest user message
    user_msgs = [m for m in req.messages if m.role == "user"]
    if not user_msgs:
        raise HTTPException(status_code=400, detail="No user message found")

    question = user_msgs[-1].content
    prompt   = retrieve_and_answer(question)
    req_id   = f"chatcmpl-{uuid.uuid4().hex[:8]}"
    created  = int(time.time())

    if req.stream:
        def streamer():
            for chunk in ollama.chat(
                model=ANSWER_MODEL,
                messages=[{"role": "user", "content": prompt}],
                stream=True
            ):
                token = chunk["message"]["content"]
                data  = {
                    "id": req_id, "object": "chat.completion.chunk",
                    "created": created, "model": "rag-llama3.2",
                    "choices": [{"index": 0, "delta": {"content": token}, "finish_reason": None}]
                }
                yield f"data: {json.dumps(data)}\\n\\n"
            yield f"data: {json.dumps({\"choices\": [{\"finish_reason\": \"stop\"}]})}\\n\\n"
            yield "data: [DONE]\\n\\n"
        return StreamingResponse(streamer(), media_type="text/event-stream")
    else:
        response = ollama.chat(
            model=ANSWER_MODEL,
            messages=[{"role": "user", "content": prompt}]
        )
        answer = response["message"]["content"]
        return {
            "id": req_id, "object": "chat.completion", "created": created,
            "model": "rag-llama3.2",
            "choices": [{"index": 0, "message": {"role": "assistant", "content": answer}, "finish_reason": "stop"}],
            "usage": {"prompt_tokens": len(prompt)//4, "completion_tokens": len(answer)//4, "total_tokens": (len(prompt)+len(answer))//4}
        }

@app.get("/health")
def health():
    return {"status": "ok", "text_chunks": text_collection.count(), "image_chunks": image_collection.count()}

if __name__ == "__main__":
    import uvicorn
    print("\\n🚀 RAG API server starting...")
    print(f"   URL: http://127.0.0.1:8100")
    print(f"   For Open WebUI: http://127.0.0.1:8100/v1")
    print(f"   Model name: rag-llama3.2")
    uvicorn.run(app, host="0.0.0.0", port=8100, log_level="warning")
'''

# Write server to file
with open("rag_server.py", "w") as f:
    f.write(server_code)

print("✅ rag_server.py written!")
print("""
╔══════════════════════════════════════════════════════════════╗
║  To start the API server, open a NEW terminal and run:       ║
║                                                              ║
║      python rag_server.py                                    ║
║                                                              ║
║  Then in Open WebUI:                                         ║
║  Settings → Connections → Add OpenAI connection:             ║
║    URL:     http://127.0.0.1:8100/v1                         ║
║    API Key: rag-key                                          ║
║    Model:   rag-llama3.2                                     ║
╚══════════════════════════════════════════════════════════════╝
""")

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 812-813: truncated \UXXXXXXXX escape (1934647681.py, line 178)

## 🔄 CELL 13 — Re-ingest New Documents

When you add new PDFs to your documents folder, run this cell to add them to ChromaDB **without re-processing existing ones**.

In [ ]:
def reingest_new_documents():
    """
    Incremental ingestion: only processes PDFs not already in ChromaDB.
    Checks existing document IDs to avoid duplicates.
    """
    # Get list of already-ingested sources
    existing = text_collection.get(limit=10000)
    ingested_sources = set()
    if existing["metadatas"]:
        ingested_sources = {m["source"] for m in existing["metadatas"]}

    # Find new PDFs
    all_pdfs = set(p.name for p in Path(DOCS_PATH).glob("*.pdf"))
    new_pdfs = all_pdfs - ingested_sources

    if not new_pdfs:
        print("✅ No new documents to ingest. All PDFs are already in ChromaDB.")
        print(f"   Ingested: {ingested_sources}")
        return

    print(f"Found {len(new_pdfs)} new document(s): {new_pdfs}")
    print("Starting ingestion...")

    for pdf_name in new_pdfs:
        pdf_path = Path(DOCS_PATH) / pdf_name
        print(f"\nProcessing: {pdf_name}")

        # Text chunks
        pages  = extract_text_from_pdf(pdf_path)
        chunks = []
        for page in pages:
            chunks.extend(chunk_page(page))

        # Table chunks
        table_cks = extract_tables_from_pdf(pdf_path)

        # Embed and store text + tables
        for i, chunk in enumerate(tqdm(chunks + table_cks, desc=f"Embedding {pdf_name}")):
            emb = get_mxbai_embedding(chunk["content"])
            if emb:
                chunk_id = f"{chunk['type']}_{pdf_name}_{chunk['page']}_{i}"
                text_collection.add(
                    ids=[chunk_id],
                    embeddings=[emb],
                    documents=[chunk["content"]],
                    metadatas=[{"page": chunk["page"], "type": chunk["type"], "source": chunk["source"]}]
                )

        # Image chunks
        img_cks = extract_images_from_pdf(pdf_path, IMAGES_PATH)
        for i, chunk in enumerate(img_cks):
            img_id = f"img_{pdf_name}_{chunk['page']}_{i}_new"
            image_collection.add(
                ids=[img_id],
                embeddings=[chunk["clip_embedding"]],
                documents=[chunk["content"]],
                metadatas=[{"page": chunk["page"], "type": "image", "source": chunk["source"], "image_path": chunk["image_path"]}]
            )

        print(f"   ✅ Done: {pdf_name}")

    print(f"""
🎉 Re-ingestion complete!
   text_and_tables : {text_collection.count()} chunks
   images          : {image_collection.count()} chunks
""")


# Run to check for new documents
reingest_new_documents()

## 💬 CELL 14 — Interactive Chat Interface

A simple chat loop you can use directly in the notebook.

In [ ]:
def chat_loop():
    """
    Simple interactive chat in the notebook.
    Type 'quit' or 'exit' to stop.
    """
    print("╔══════════════════════════════════════════════════════╗")
    print("║  📚 Document RAG Chat                                ║")
    print("║  Ask anything about your documents.                  ║")
    print("║  Type 'quit' to exit.                                ║")
    print("╚══════════════════════════════════════════════════════╝")

    while True:
        try:
            question = input("\n🙋 You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nChat ended.")
            break

        if not question:
            continue
        if question.lower() in ["quit", "exit", "bye"]:
            print("Goodbye!")
            break

        ask(question, stream=True)


# Start the chat
chat_loop()

## 📋 CELL 15 — System Status Dashboard

In [ ]:
def print_status():
    """Print a summary of the current RAG system state."""
    print("╔══════════════════════════════════════════════════════════════╗")
    print("║              📊 RAG SYSTEM STATUS                           ║")
    print("╠══════════════════════════════════════════════════════════════╣")

    # ChromaDB stats
    tc = text_collection.count()
    ic = image_collection.count()
    print(f"║  🗄️  ChromaDB                                                ║")
    print(f"║     text_and_tables collection : {tc:>6} chunks              ║")
    print(f"║     images collection          : {ic:>6} chunks              ║")
    print(f"║     Total                      : {tc+ic:>6} chunks              ║")

    # Document breakdown
    print(f"║                                                              ║")
    print(f"║  📁 Documents in ChromaDB                                    ║")
    if tc > 0:
        existing = text_collection.get(limit=10000)
        sources = {}
        for m in existing["metadatas"]:
            src = m["source"]
            typ = m["type"]
            if src not in sources:
                sources[src] = {"text": 0, "table": 0}
            sources[src][typ] = sources[src].get(typ, 0) + 1
        for src, counts in sources.items():
            t_cnt = counts.get("text", 0)
            tb_cnt = counts.get("table", 0)
            line = f"     • {src[:35]:<35} {t_cnt}t {tb_cnt}tab"
            print(f"║  {line:<60}║")

    # Model status
    print(f"║                                                              ║")
    print(f"║  🤖 Models                                                   ║")
    print(f"║     Text embedding  : mxbai-embed-large (1024-dim)           ║")
    print(f"║     Image embedding : CLIP ViT-B/32 (512-dim) on {DEVICE:<5}    ║")
    print(f"║     Reranker        : MiniLM-L-6-v2 CrossEncoder             ║")
    print(f"║     Answer LLM      : {ANSWER_MODEL:<38}║")

    # API
    print(f"║                                                              ║")
    print(f"║  🌐 API Server (run rag_server.py in terminal)               ║")
    print(f"║     Endpoint : http://127.0.0.1:{API_PORT}/v1                    ║")
    print(f"║     Model ID : rag-llama3.2                                  ║")
    print("╚══════════════════════════════════════════════════════════════╝")

print_status()